# Ridge Regression — Mathematics

**Goal.** Derive ridge regression from scratch as the OLS loss plus an L^2-norm penalty on the coefficients. Prove that the resulting problem is **always uniquely solvable** (even when OLS is singular), derive the closed-form solution, decompose it in the SVD basis to see *exactly* how each component is shrunk, prove equivalence to a constrained least-squares problem, and connect ridge to Bayesian MAP estimation under a Gaussian prior.

**Role of this notebook.** Pure mathematics — definitions, derivations, theorems. No code, no plots. Intuition is in `01_intuition.ipynb`; algorithms in `03_optimization.ipynb`; implementation in `05_hands_on_programming.ipynb`.

**Prerequisites.** `01_linear_regression/02_mathematics.ipynb` — closed-form OLS, the matrix calculus identities, the SVD (Theorem 6.1) and the Moore–Penrose pseudoinverse. We reuse all of it.

**Stage map.** `01_intuition` → **`02_mathematics`** → `03_optimization` → `04_statistics` → `05_hands_on_programming`.

**Six questions.**

1. What exactly is the ridge loss, and what does the new term *mean*?
2. Why is the optimisation problem always uniquely solvable for $\lambda > 0$?
3. What is the closed-form solution $\hat{\theta}_{\text{ridge}}$?
4. How does ridge act on each SVD component of the data?
5. Is ridge equivalent to a constrained problem? (Yes — Lagrangian view.)
6. What probabilistic prior produces ridge as a MAP estimator?

---

**Reading conventions.** Same as `01_linear_regression/02_mathematics.ipynb`: theorem statements in blockquotes, multi-line derivations in code blocks, equations numbered when referenced later.


## 0. Notation (additions to `01_linear_regression` conventions)

We reuse everything from `01_linear_regression/02_mathematics.ipynb` §0 and add:

| Symbol | Type | Meaning |
|---|---|---|
| $\lambda$ | scalar $\ge$ 0 | **regularisation strength** (a.k.a. ridge parameter, Tikhonov parameter) |
| $L_{\text{ridge}}(\theta)$ | scalar function | the penalised loss; $L_{\text{ridge}} : \mathbb{R}^p \to \mathbb{R}^+$ |
| $\hat{\theta}_{\text{ridge}}$ | vector $\in \mathbb{R}^p$ | the unique minimiser of L_ridge for $\lambda > 0$ |
| $I_p$ | matrix $\in \mathbb{R}^{p \times p}$ | identity matrix of size p |
| $\sigma_i$ | scalar | singular values of $X$ (`01_linear_regression/02_mathematics.ipynb` Theorem 6.1) |
| t | scalar > 0 | constraint radius in the equivalent constrained problem (§5) |

**Important convention (intercept).** In the *clean mathematical* treatment we penalise *all* coefficients including the intercept. In *practice* (`05_hands_on_programming.ipynb`) the intercept is left unpenalised — otherwise shifting y by a constant changes the fit. The standard programming trick is to centre y and the columns of X first, then fit ridge without an intercept column, then back out the intercept as ȳ. This adjustment does not change any of the theorems below; it only changes which p the formulas use (p instead of p + 1).


## 1. The ridge loss

### 1.1 Definition

Given the OLS loss $L_{\text{OLS}}(\theta)$ = (1/n) $\cdot$ $\|X\theta - y\|^2$ (eq. 2.1 of `01_linear_regression/02_mathematics.ipynb`), the **ridge loss** with parameter $\lambda$ $\ge$ 0 is

> $$L_{\text{ridge}}(\theta) := L_{\text{OLS}}(\theta) + \lambda \|\theta\|^2 = \frac{1}{n} \|X\theta - y\|^2 + \lambda \theta^T \theta$$   (1.1)

**Ridge regression** is the optimisation problem

> $$\hat{\theta}_{\text{ridge}} \in \arg\min_{\theta \in \mathbb{R}^p} L_{\text{ridge}}(\theta)$$   (1.2)

### 1.2 Reading

- $\lambda$ = 0 — pure OLS (recovers the contents of `01_linear_regression/02_mathematics.ipynb`).
- $\lambda \to \infty$ — the penalty dominates; the unique minimum of $\theta^\top\theta$ in $\mathbb{R}^p$ is 0; $\hat{\theta}_{\text{ridge}} \to 0$.
- The factor 1/n on the data term is a convention; many books (and `sklearn.linear_model.Ridge`) use the sum-of-squares loss $\|X\theta - y\|^2$ + $\alpha \|\theta\|^2$, which is the same problem with $\alpha := n\lambda$. Multiplicative constants between L_OLS and the penalty are what *actually* matter; absolute scaling is a parametrisation choice.


## 2. The closed-form solution

### 2.1 Gradient and Hessian

The gradient and Hessian of L_OLS were computed in `01_linear_regression/02_mathematics.ipynb` (Theorems 3.2 and 3.3); the penalty contributes two extra terms via the matrix-calculus identities (3.1):

> $$\nabla L_{\text{ridge}}(\theta) = \frac{2}{n} X^T (X\theta - y) + 2\lambda \theta$$
>
> $$\nabla^2 L_{\text{ridge}}(\theta) = \frac{2}{n} X^T X + 2\lambda I_p$$   (constant in $\theta$).   (2.1)

### 2.2 Theorem (always invertible, strictly convex)

> **Theorem 2.2.** For every $\lambda > 0$ (regardless of $\text{rank}(X)$),
>
> 1. $X^\top X$ + n $\lambda$ $I_p$ is symmetric positive-definite.
> 2. L_ridge is strictly convex.
> 3. The optimisation problem (1.2) has a unique minimiser.

**Proof.**

*(1) PD.* For any v $\in$ $\mathbb{R}^p$ with $v \neq 0$,

```
$$v^\top (X^\top X + n\lambda I_p)v = \|Xv\|_2^2 + n\lambda\|v\|_2^2 \ge n\lambda\|v\|_2^2 > 0.$$ 
```

using $\lambda > 0$. (Note: the first term is only $\ge$ 0 — it can be 0 if X v = 0 — but the second term is strictly positive.)

*(2) Strict convexity.* By (2.1) and the calculation in (1), $\nabla^2 L_{\text{ridge}} = \frac{2}{n}(X^\top X + n\lambda I_p)$ is PD everywhere, so L_ridge is strictly convex (`01_linear_regression/02_mathematics.ipynb` Theorem 3.4 part 2 applies verbatim).

*(3) Unique minimum.* A strictly convex function on $\mathbb{R}^p$ that grows like $\|\theta\|^2$ at infinity attains its minimum at exactly one point. ∎

**Reading.** Theorem 2.2 is the whole reason ridge exists. OLS is uniquely solvable *only* when $\text{rank}(X)$ = p (Theorem 4.2 of the previous folder); ridge is uniquely solvable for *any* X and any $\lambda > 0$. This is what makes ridge usable on collinear and high-dimensional designs (p > n) where OLS breaks down.

### 2.3 Theorem (closed form)

> **Theorem 2.3.** For every $\lambda > 0$,
>
> $$\hat{\theta}_{\text{ridge}} = (X^T X + n \lambda I_p)^{-1} X^T y$$   (2.2)

**Proof.** First-order optimality: set $\nabla$ $L_{\text{ridge}}(\theta)$ = 0 in (2.1) and cancel (2/n):

```
$\frac{2}{n}X^\top(X\theta-y)+2\lambda\theta$  =  0
⟺  $(X^\top X)\theta - X^\top y + n\lambda\theta$  =  0
⟺  $(X^\top X + n\lambda I_p)\theta = X^\top y$.                                                  (2.3)
```

(2.3) is the **ridge normal equations**. The matrix on the left is invertible by Theorem 2.2(1), and strict convexity (Theorem 2.2(2)) makes the resulting critical point the unique global minimiser. ∎

**Comparison with OLS.** Side by side, the difference is one rank-p positive shift:

| Estimator | Closed form | Requires |
|---|---|---|
| OLS  | $(X^\top X)^{-1}X^\top y$ | $\text{rank}(X)$ = p |
| Ridge | $(X^\top X + n\lambda I_p)^{-1}X^\top y$ | $\lambda > 0$ (no rank assumption) |


## 3. SVD perspective: ridge as componentwise shrinkage

What does (2.2) *do* to the data? The cleanest view comes from substituting the SVD of $X$.

### 3.1 Setup

Let $X = U\Sigma V^\top$ be the thin SVD (`01_linear_regression/02_mathematics.ipynb` Theorem 6.1), with non-negative singular values $\sigma_1 \ge \dots \ge \sigma_p \ge 0$ on the diagonal of $\Sigma$. Then

$$
X^\top X
= V\Sigma^\top U^\top U\Sigma V^\top
= V(\Sigma^\top\Sigma)V^\top
= V\operatorname{diag}(\sigma_1^2, \dots, \sigma_p^2)V^\top.
$$

Similarly,

$$
X^\top X + n\lambda I_p
= V\operatorname{diag}(\sigma_j^2 + n\lambda)V^\top.
$$

### 3.2 Theorem (SVD shrinkage formula)

> **Theorem 3.2.** Let $X = U\Sigma V^\top$ and let $u_j, v_j$ denote the $j$-th columns of $U$ and $V$. Then
>
> $$\hat{\theta}_{\text{ridge}} = \sum_{j=1}^p \frac{\sigma_j}{\sigma_j^2 + n\lambda}\langle u_j, y \rangle v_j.$$   (3.1)

**Proof.** Apply (2.2):

$$
\begin{aligned}
\hat{\theta}_{\text{ridge}}
&= (X^\top X + n\lambda I_p)^{-1}X^\top y \\
&= V\operatorname{diag}\left(\frac{1}{\sigma_j^2+n\lambda}\right)V^\top V\Sigma^\top U^\top y \\
&= V\operatorname{diag}\left(\frac{\sigma_j}{\sigma_j^2+n\lambda}\right)U^\top y.
\end{aligned}
$$

Reading off this product columnwise gives (3.1). ∎

### 3.3 Theorem (predictions as filtered projection)

> **Theorem 3.3.** Let
>
> $$H_\lambda := X(X^\top X+n\lambda I_p)^{-1}X^\top \in \mathbb{R}^{n\times n}$$
>
> be the **ridge hat matrix**. Then
>
> $$H_\lambda = \sum_{j=1}^p \frac{\sigma_j^2}{\sigma_j^2+n\lambda}u_j u_j^\top,$$   (3.2)
>
> and $\hat{y}_{\text{ridge}} = H_\lambda y$.

**Proof.** Substitute $X = U\Sigma V^\top$ into the definition of $H_\lambda$. The $V$ terms cancel as in §3.2, giving

$$
H_\lambda = U\Sigma(\Sigma^\top\Sigma+n\lambda I_p)^{-1}\Sigma^\top U^\top.
$$

The middle factor is diagonal with entry $\sigma_j^2/(\sigma_j^2+n\lambda)$, which gives (3.2). ∎

### 3.4 Reading: differential shrinkage

Define the **shrinkage factor** for SVD component $j$:

$$
\rho_j(\lambda) := \frac{\sigma_j^2}{\sigma_j^2+n\lambda} \in [0,1].
$$

Three immediate consequences:

1. **OLS limit.** $\rho_j(0)=1$ for $\sigma_j>0$: no shrinkage.
2. **Extreme regularisation.** $\rho_j(\infty)=0$: every component is shrunk to zero, so $\hat{y}_{\text{ridge}} \to 0$.
3. **Small singular values shrink most.** For fixed $\lambda$, $\rho_j(\lambda)$ is small when $\sigma_j^2 \ll n\lambda$.

This is exactly the right behaviour: low-singular-value directions are also the directions with high variance in $\hat{\theta}_{\text{OLS}}$, since $\operatorname{Var}(\hat{\theta}_{\text{OLS}})=\sigma^2(X^\top X)^{-1}$ has eigenvalues $\sigma^2/\sigma_j^2$.

**Picture.** OLS projects $y$ orthogonally onto $\operatorname{Col}(X)$. Ridge replaces each projection eigenvalue $1$ by $\sigma_j^2/(\sigma_j^2+n\lambda)$, a smooth roll-off. Tikhonov regularisation is a filtered SVD.


## 4. Effective degrees of freedom

OLS has p parameters; $\text{trace}(H)$ = p (`01_linear_regression/02_mathematics.ipynb` Theorem 5.3 part 4). Ridge has *the same* p coefficients but uses them less freely. The **effective degrees of freedom** quantifies this.

### 4.1 Definition

> $$\text{df}(\lambda) := \text{trace}(H_\lambda) = \sum_{j=1}^p \frac{\sigma_j^2}{\sigma_j^2 + n\lambda}$$   (4.1)

### 4.2 Reading

- $\lambda$ = 0 ⇒ each $\rho_j$ = 1 (when $\sigma_j$ > 0) ⇒ df = $\text{rank}(X)$ — the OLS count of effective parameters.
- $\lambda \to \infty$ ⇒ each $\rho_j$ → 0 ⇒ df → 0 — the model has no effective freedom; predictions collapse to a constant.
- For intermediate $\lambda$, df($\lambda$) is a continuous, *monotone-decreasing* function of $\lambda$ that interpolates smoothly between the two endpoints.

**Why this matters.** df($\lambda$) is the right quantity to substitute for p when computing residual standard errors, AIC, BIC, and the LOO-CV closed form — it is the *true* number of parameters "used" by the fit. `04_statistics.ipynb` uses df($\lambda$) extensively.


## 5. Equivalence to a constrained problem

Ridge can be stated in two equivalent forms: *penalty* and *constraint*.

### 5.1 Theorem (penalty ⇔ constraint duality)

> **Theorem 5.1.** For every $t>0$ there exists a unique $\lambda(t)\ge 0$ such that
>
> $$\arg\min_\theta \frac{1}{n}\|X\theta-y\|_2^2 \quad \text{subject to}\quad \|\theta\|_2^2\le t
> \quad = \quad
> \arg\min_\theta L_{\text{ridge}}(\theta;\lambda(t)).$$   (5.1)
>
> The map $t \leftrightarrow \lambda(t)$ is monotone decreasing: smaller budget $t$ means larger penalty $\lambda$.

**Proof sketch.** The Lagrangian of the constrained problem is

$$
\mathcal{L}(\theta,\lambda)
= \frac{1}{n}\|X\theta-y\|_2^2 + \lambda(\|\theta\|_2^2-t).
$$

The KKT conditions force $\lambda\ge 0$ and $\lambda(\|\theta\|_2^2-t)=0$ (complementary slackness). If the OLS solution already satisfies $\|\theta^*_{\text{OLS}}\|_2^2\le t$, the constraint is inactive and $\lambda=0$. Otherwise it is active and $\lambda$ is chosen so that $\|\hat{\theta}\|_2^2=t$. Solving $\nabla_\theta\mathcal{L}=0$ reproduces the ridge normal equations (2.3). ∎

### 5.2 Reading

**Geometric picture.** The penalty form draws iso-loss contours of $\frac{1}{n}\|X\theta-y\|_2^2$ and iso-penalty contours of $\|\theta\|_2^2$. The ridge solution is the first tangency between an ellipsoid and a sphere.

Because the constraint region is a ball, tangency happens at a generic point. Ridge therefore shrinks coefficients continuously but does not usually set them exactly to zero. Lasso differs because the $\ell_1$ constraint is a diamond with corners on the axes.


## 6. Bayesian interpretation: MAP under a Gaussian prior

OLS = MLE under Gaussian noise (`01_linear_regression/02_mathematics.ipynb` Theorem 2.2). Ridge = **MAP** under Gaussian noise and a Gaussian prior on $\theta$.

### 6.1 Theorem (ridge as Gaussian MAP)

> **Theorem 6.1.** Assume the data model
>
> $$y \mid X, \theta \sim \mathcal{N}(X\theta, \sigma^2 I_n)$$
>
> together with the prior
>
> $$\theta \sim \mathcal{N}(0, \tau^2 I_p).$$
>
> Then the maximum a posteriori estimator of $\theta$ is the ridge estimator (2.2) with
>
> $$\lambda = \frac{\sigma^2}{n\tau^2}.$$

**Proof.** Bayes' rule gives

$$
p(\theta \mid y) \propto p(y \mid \theta)p(\theta).
$$

Taking negative log and dropping additive constants that do not depend on $\theta$:

$$
-\log p(\theta \mid y)
= \frac{1}{2\sigma^2}\|y-X\theta\|_2^2 + \frac{1}{2\tau^2}\|\theta\|_2^2 + \text{constant}.
$$

Multiplying by $2\sigma^2/n$ does not change the optimizer:

$$
-\frac{2\sigma^2}{n}\log p(\theta \mid y)
= \frac{1}{n}\|y-X\theta\|_2^2 + \frac{\sigma^2}{n\tau^2}\|\theta\|_2^2 + \text{constant}.
$$

The right-hand side is $L_{\text{ridge}}(\theta;\lambda=\sigma^2/(n\tau^2))$ plus a constant. Therefore

$$
\arg\max_\theta p(\theta \mid y) = \arg\min_\theta L_{\text{ridge}}(\theta).
$$

∎

### 6.2 Reading

- Small $\tau^2$ says "I believe $\theta$ is near zero", which implies large $\lambda$ and strong shrinkage.
- Large $\tau^2$ says "I have weak prior information", which implies small $\lambda$ and an almost-OLS fit.
- Noisier data increases $\sigma^2$, which increases $\lambda$; more data increases $n$, which decreases $\lambda$.


## Takeaway

- **Loss.** $L_{\text{ridge}}(\theta)=\frac{1}{n}\|X\theta-y\|_2^2+\lambda\|\theta\|_2^2$ (eq. 1.1). One new knob: $\lambda\ge 0$.
- **Existence (Theorem 2.2).** $X^\top X+n\lambda I_p$ is positive definite for any $X$ and any $\lambda>0$. Ridge is strictly convex and has a unique minimizer.
- **Closed form (Theorem 2.3).** $\hat{\theta}_{\text{ridge}}=(X^\top X+n\lambda I_p)^{-1}X^\top y$ (eq. 2.2). No rank assumption on $X$ is required.
- **SVD shrinkage (Theorem 3.2-3.3).** Each SVD component is multiplied by $\sigma_j^2/(\sigma_j^2+n\lambda)\in(0,1]$. Small singular values are shrunk most.
- **Effective dof (eq. 4.1).** $\operatorname{df}(\lambda)=\sum_j \sigma_j^2/(\sigma_j^2+n\lambda)$, decreasing from $\operatorname{rank}(X)$ to $0$ as $\lambda$ grows.
- **Constrained form (Theorem 5.1).** Ridge is equivalent to OLS subject to $\|\theta\|_2^2\le t$.
- **Bayesian view (Theorem 6.1).** Ridge is MAP under Gaussian likelihood and Gaussian prior, with $\lambda=\sigma^2/(n\tau^2)$.

Next: `03_optimization.ipynb` — the closed form is useful when $p$ is small; for large $p$, gradient descent uses the same gradient with a one-line regularization modification.
